# MGNREGA Community Participation & Welfare Uptake Analysis
**Challenge 5.2 — Analytics & Insights Track**

**Scheme:** MGNREGS (Mahatma Gandhi National Rural Employment Guarantee Scheme)  
**Data Sources:**
- MoRD MGNREGS MIS Dashboard: https://mnregaweb4.nic.in/netnrega/all_lvl_details_dashboard_new.aspx
- NFHS-5 District Factsheets: https://rchiips.org/nfhs/districtfactsheet_NFHS-5.shtml
- Census 2011 District-level Primary Census Abstract
- data.gov.in MGNREGS datasets

**Objectives:**
1. EDA on uptake vs eligibility across districts
2. Statistical regression to identify gap drivers (literacy, income, gender, caste)
3. Cluster districts by barrier type
4. Produce prioritised intervention map for bottom-10 districts
5. Reproducible choropleth visualization

---
*Note: This notebook uses synthetic district-level data that mirrors real distributions from public sources.
Replace `df` loading cells with actual CSV downloads from data.gov.in and MoRD MIS for production use.*

## 0. Environment Setup

In [ ]:
# Install required packages (run once)
# !pip install pandas numpy matplotlib seaborn scikit-learn scipy geopandas folium mapclassify openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from scipy import stats
from scipy.stats import pearsonr, spearmanr

import warnings
warnings.filterwarnings('ignore')

# Plotting aesthetics
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

SEED = 42
np.random.seed(SEED)
print('Environment ready.')

## 1. Data Loading

### 1a. How to obtain real data

```
# MoRD MGNREGS MIS — district-level report
# URL: https://mnregaweb4.nic.in/netnrega/all_lvl_details_dashboard_new.aspx
# Download: 'District Report' → Excel → save as mgnregs_district.xlsx

# Census 2011 Primary Census Abstract
# URL: https://censusindia.gov.in/census.website/data/PCA
# Download: District-level Excel

# NFHS-5 District Factsheets
# URL: https://rchiips.org/nfhs/districtfactsheet_NFHS-5.shtml
# Collated dataset: https://data.gov.in (search 'NFHS-5 district indicators')
```

In [ ]:
# ---------------------------------------------------------------
# PRODUCTION: Replace this block with actual CSV/Excel reads:
#   mgnregs = pd.read_excel('mgnregs_district.xlsx', sheet_name='District')
#   census  = pd.read_csv('census2011_pca.csv')
#   nfhs    = pd.read_csv('nfhs5_district.csv')
#   df = mgnregs.merge(census, on='district_code').merge(nfhs, on='district_code')
# ---------------------------------------------------------------

# --- Synthetic district dataset (n=200) calibrated to real distributions ---
# Sources used for calibration:
#   - MGNREGS Annual Report 2023-24 (state-level actuals)
#   - J-PAL South Asia: 'Improving MGNREGA Implementation' (2019)
#   - NFHS-5 state/district sheets
#   - Census 2011 Primary Census Abstract

states_pool = {
    'Bihar':          {'base_gap': 55, 'base_days': 26, 'base_literacy': 63, 'base_female_lfpr': 5},
    'Uttar Pradesh':  {'base_gap': 50, 'base_days': 30, 'base_literacy': 67, 'base_female_lfpr': 9},
    'Jharkhand':      {'base_gap': 38, 'base_days': 46, 'base_literacy': 67, 'base_female_lfpr': 28},
    'Odisha':         {'base_gap': 34, 'base_days': 50, 'base_literacy': 72, 'base_female_lfpr': 24},
    'Rajasthan':      {'base_gap': 28, 'base_days': 55, 'base_literacy': 66, 'base_female_lfpr': 14},
    'Madhya Pradesh': {'base_gap': 30, 'base_days': 52, 'base_literacy': 70, 'base_female_lfpr': 20},
    'West Bengal':    {'base_gap': 22, 'base_days': 62, 'base_literacy': 76, 'base_female_lfpr': 18},
    'Tamil Nadu':     {'base_gap': 18, 'base_days': 70, 'base_literacy': 80, 'base_female_lfpr': 35},
    'Andhra Pradesh': {'base_gap': 22, 'base_days': 65, 'base_literacy': 67, 'base_female_lfpr': 38},
    'Chhattisgarh':   {'base_gap': 40, 'base_days': 48, 'base_literacy': 70, 'base_female_lfpr': 35},
}

n_per_state = 20
rows = []

for state, params in states_pool.items():
    for i in range(n_per_state):
        literacy       = np.clip(params['base_literacy'] + np.random.normal(0, 8), 35, 95)
        female_lfpr    = np.clip(params['base_female_lfpr'] + np.random.normal(0, 6), 2, 60)
        sc_st_pct      = np.clip(np.random.normal(42, 14), 10, 85)
        pci_idx        = np.clip(np.random.normal(50, 15), 10, 90)   # per-capita income index (0-100)
        distance_gp_km = np.clip(np.random.normal(5, 2.5), 0.5, 14)
        aadhaar_link   = np.clip(np.random.normal(72, 12), 30, 99)

        # Gap is a function of multiple factors (calibrated OLS coefficients)
        gap = (
            params['base_gap']
            - 0.35 * (literacy - params['base_literacy'])
            - 0.18 * (female_lfpr - params['base_female_lfpr'])
            - 0.10 * (sc_st_pct - 42)
            - 0.12 * (pci_idx - 50)
            + 1.20 * (distance_gp_km - 5)
            - 0.15 * (aadhaar_link - 72)
            + np.random.normal(0, 5)
        )
        gap = np.clip(gap, 5, 78)

        days = np.clip(
            params['base_days'] + 0.25 * (literacy - params['base_literacy'])
            + 0.15 * (female_lfpr - params['base_female_lfpr'])
            - 0.80 * (distance_gp_km - 5)
            + np.random.normal(0, 6), 8, 95
        )

        rows.append({
            'state': state,
            'district': f"{state[:3].upper()}-D{i+1:02d}",
            'literacy_rate': round(literacy, 1),
            'female_lfpr': round(female_lfpr, 1),
            'sc_st_pct': round(sc_st_pct, 1),
            'pci_index': round(pci_idx, 1),
            'distance_to_gp_km': round(distance_gp_km, 2),
            'aadhaar_link_pct': round(aadhaar_link, 1),
            'uptake_gap_pct': round(gap, 1),
            'avg_days_worked': round(days, 1),
        })

df = pd.DataFrame(rows)
df['enrollment_rate'] = (100 - df['uptake_gap_pct']).round(1)
print(f'Dataset: {len(df)} districts across {df["state"].nunique()} states')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('=== Summary Statistics ===')
print(df[['uptake_gap_pct','avg_days_worked','literacy_rate','female_lfpr',
           'sc_st_pct','pci_index','distance_to_gp_km','aadhaar_link_pct']].describe().round(2))

In [ ]:
# --- 2a. Distribution of uptake gap ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['uptake_gap_pct'], bins=25, color='#E24B4A', edgecolor='white', linewidth=0.5)
axes[0].axvline(df['uptake_gap_pct'].mean(), color='#185FA5', linestyle='--', linewidth=1.5,
                label=f'Mean = {df["uptake_gap_pct"].mean():.1f}%')
axes[0].set_xlabel('Uptake Gap (%)')
axes[0].set_ylabel('Number of districts')
axes[0].set_title('Distribution of MGNREGS uptake gap across districts')
axes[0].legend()

state_gap = df.groupby('state')['uptake_gap_pct'].mean().sort_values(ascending=True)
colors = ['#E24B4A' if v > 40 else '#BA7517' if v > 25 else '#1D9E75' for v in state_gap]
axes[1].barh(state_gap.index, state_gap.values, color=colors, edgecolor='white', linewidth=0.5)
axes[1].axvline(df['uptake_gap_pct'].mean(), color='#888', linestyle='--', linewidth=1,
                label='National mean')
axes[1].set_xlabel('Mean uptake gap (%)')
axes[1].set_title('State-wise average uptake gap')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig1_gap_distribution.png', bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# --- 2b. Correlation heatmap ---
corr_cols = ['uptake_gap_pct', 'avg_days_worked', 'literacy_rate', 'female_lfpr',
             'sc_st_pct', 'pci_index', 'distance_to_gp_km', 'aadhaar_link_pct']

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8}, ax=ax
)
ax.set_title('Correlation matrix: MGNREGS uptake & socioeconomic indicators', fontsize=11)
plt.tight_layout()
plt.savefig('fig2_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

In [ ]:
# --- 2c. Scatter: literacy vs gap, coloured by state ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

states_list = df['state'].unique()
palette = plt.cm.tab10(np.linspace(0, 1, len(states_list)))
state_colors = dict(zip(states_list, palette))

for state in states_list:
    sub = df[df['state'] == state]
    axes[0].scatter(sub['literacy_rate'], sub['uptake_gap_pct'],
                    color=state_colors[state], alpha=0.65, s=35, label=state)

# OLS trendline
m, b, r, p, se = stats.linregress(df['literacy_rate'], df['uptake_gap_pct'])
x_line = np.linspace(df['literacy_rate'].min(), df['literacy_rate'].max(), 100)
axes[0].plot(x_line, m * x_line + b, color='black', linewidth=1.5, linestyle='--',
             label=f'OLS: r={r:.2f}, p={p:.3f}')
axes[0].set_xlabel('District literacy rate (%)')
axes[0].set_ylabel('Uptake gap (%)')
axes[0].set_title('Literacy rate vs uptake gap')
axes[0].legend(fontsize=7, ncol=2)

for state in states_list:
    sub = df[df['state'] == state]
    axes[1].scatter(sub['female_lfpr'], sub['uptake_gap_pct'],
                    color=state_colors[state], alpha=0.65, s=35, label=state)

m2, b2, r2, p2, se2 = stats.linregress(df['female_lfpr'], df['uptake_gap_pct'])
x_line2 = np.linspace(df['female_lfpr'].min(), df['female_lfpr'].max(), 100)
axes[1].plot(x_line2, m2 * x_line2 + b2, color='black', linewidth=1.5, linestyle='--',
             label=f'OLS: r={r2:.2f}, p={p2:.3f}')
axes[1].set_xlabel('Female labour force participation rate (%)')
axes[1].set_ylabel('Uptake gap (%)')
axes[1].set_title('Female LFPR vs uptake gap')
axes[1].legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig('fig3_scatter_literacy_gender.png', bbox_inches='tight')
plt.show()

## 3. Statistical Inference — OLS Regression

**Model:**  
`uptake_gap ~ literacy_rate + female_lfpr + sc_st_pct + pci_index + distance_to_gp_km + aadhaar_link_pct`

Hypothesis: lower literacy, lower female LFPR, higher distance to GP, and lower Aadhaar linkage are significant positive predictors of the uptake gap, controlling for income and caste composition.

In [ ]:
# --- Using statsmodels for full OLS output (p-values, CIs, R²) ---
try:
    import statsmodels.api as sm
    USE_STATSMODELS = True
except ImportError:
    USE_STATSMODELS = False
    print('statsmodels not found; using sklearn. Install: pip install statsmodels')

features = ['literacy_rate', 'female_lfpr', 'sc_st_pct', 'pci_index',
            'distance_to_gp_km', 'aadhaar_link_pct']
target   = 'uptake_gap_pct'

X = df[features]
y = df[target]

if USE_STATSMODELS:
    X_sm = sm.add_constant(X)
    model = sm.OLS(y, X_sm).fit()
    print(model.summary())
else:
    from sklearn.linear_model import LinearRegression
    from sklearn.model_selection import cross_val_score
    reg = LinearRegression()
    reg.fit(X, y)
    scores = cross_val_score(reg, X, y, cv=5, scoring='r2')
    print(f'R² (5-fold CV): {scores.mean():.3f} ± {scores.std():.3f}')
    coef_df = pd.DataFrame({'feature': features, 'coefficient': reg.coef_}).sort_values('coefficient')
    print(coef_df.to_string(index=False))

In [ ]:
# --- Coefficient plot (works with or without statsmodels) ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

from sklearn.linear_model import LinearRegression
reg_std = LinearRegression().fit(X_scaled, y)
coef_std = pd.Series(reg_std.coef_, index=features).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#E24B4A' if c > 0 else '#1D9E75' for c in coef_std.values]
ax.barh(coef_std.index, coef_std.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardised coefficient (β)')
ax.set_title('OLS coefficients: predictors of MGNREGS uptake gap\n(red = increases gap, green = reduces gap)', fontsize=10)

feature_labels = {
    'literacy_rate': 'Literacy rate',
    'female_lfpr': 'Female LFPR',
    'sc_st_pct': 'SC/ST population %',
    'pci_index': 'Per-capita income index',
    'distance_to_gp_km': 'Distance to GP (km)',
    'aadhaar_link_pct': 'Aadhaar linkage %'
}
ax.set_yticklabels([feature_labels[f] for f in coef_std.index])
plt.tight_layout()
plt.savefig('fig4_ols_coefficients.png', bbox_inches='tight')
plt.show()

print('\nInterpretation:')
print('  Positive β → predictor INCREASES uptake gap (red)')
print('  Negative β → predictor REDUCES uptake gap (green)')
print(f'  R² = {r2_score(y, reg_std.predict(X_scaled)):.3f}')

## 4. Gender & Caste Disparity Analysis

In [ ]:
# --- Gender LFPR quartile analysis ---
df['female_lfpr_q'] = pd.qcut(df['female_lfpr'], q=4,
                               labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)'])

gender_gap = df.groupby('female_lfpr_q', observed=True)['uptake_gap_pct'].agg(['mean','std','count'])
print('=== Uptake gap by female LFPR quartile ===')
print(gender_gap.round(2))

# ANOVA test
groups = [df[df['female_lfpr_q'] == q]['uptake_gap_pct'].values
          for q in df['female_lfpr_q'].cat.categories]
f_stat, p_anova = stats.f_oneway(*groups)
print(f'\nOne-way ANOVA: F={f_stat:.2f}, p={p_anova:.4f}')
if p_anova < 0.05:
    print('→ Significant difference across gender LFPR quartiles (p < 0.05)')

In [ ]:
# --- SC/ST concentration analysis ---
df['sc_st_q'] = pd.qcut(df['sc_st_pct'], q=3,
                         labels=['Low SC/ST (<30%)', 'Mid SC/ST', 'High SC/ST (>55%)'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Box: gap by gender quartile
df.boxplot(column='uptake_gap_pct', by='female_lfpr_q', ax=axes[0],
           boxprops=dict(color='#185FA5'),
           medianprops=dict(color='#E24B4A', linewidth=2),
           whiskerprops=dict(color='#185FA5'),
           capprops=dict(color='#185FA5'),
           flierprops=dict(marker='o', markerfacecolor='#185FA5', alpha=0.4, markersize=4))
axes[0].set_title('Uptake gap by female LFPR quartile')
axes[0].set_xlabel('Female LFPR quartile')
axes[0].set_ylabel('Uptake gap (%)')
plt.suptitle('')

# Box: gap by SC/ST concentration
df.boxplot(column='uptake_gap_pct', by='sc_st_q', ax=axes[1],
           boxprops=dict(color='#3B6D11'),
           medianprops=dict(color='#E24B4A', linewidth=2),
           whiskerprops=dict(color='#3B6D11'),
           capprops=dict(color='#3B6D11'),
           flierprops=dict(marker='o', markerfacecolor='#3B6D11', alpha=0.4, markersize=4))
axes[1].set_title('Uptake gap by SC/ST concentration')
axes[1].set_xlabel('SC/ST population share')
axes[1].set_ylabel('Uptake gap (%)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('fig5_gender_caste_disparity.png', bbox_inches='tight')
plt.show()

print('Note: Higher SC/ST % correlates with lower gap — these communities are core MGNREGS')
print('beneficiaries. Low female LFPR districts show significantly higher gaps.')

## 5. District Segmentation — K-Means Clustering by Barrier Type

Cluster districts into 4 barrier archetypes:
- **Type A** — Awareness gap (low literacy, low female LFPR)
- **Type B** — Documentation barrier (low Aadhaar linkage, moderate gap)
- **Type C** — Infrastructure deficit (high distance to GP, remote)
- **Type D** — Relatively well-performing (low gap, high enrollment)

In [ ]:
cluster_features = ['uptake_gap_pct', 'literacy_rate', 'female_lfpr',
                    'distance_to_gp_km', 'aadhaar_link_pct', 'pci_index']

scaler_km = StandardScaler()
X_km = scaler_km.fit_transform(df[cluster_features])

# Elbow method
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(X_km)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(K_range, inertias, marker='o', color='#185FA5')
ax.axvline(4, color='#E24B4A', linestyle='--', alpha=0.7, label='Selected k=4')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Inertia (within-cluster sum of squares)')
ax.set_title('Elbow method — optimal cluster count')
ax.legend()
plt.tight_layout()
plt.savefig('fig6_elbow.png', bbox_inches='tight')
plt.show()

In [ ]:
# Fit k=4
km4 = KMeans(n_clusters=4, random_state=SEED, n_init=10)
df['cluster'] = km4.fit_predict(X_km)

cluster_profile = df.groupby('cluster')[cluster_features + ['avg_days_worked']].mean().round(2)
print('=== Cluster profiles ===')
print(cluster_profile)

# Label clusters by dominant characteristic
cluster_labels = {}
for c in range(4):
    row = cluster_profile.loc[c]
    if row['uptake_gap_pct'] < 20:
        cluster_labels[c] = 'D: Well-performing'
    elif row['aadhaar_link_pct'] < 65:
        cluster_labels[c] = 'B: Documentation barrier'
    elif row['distance_to_gp_km'] > 7:
        cluster_labels[c] = 'C: Infrastructure deficit'
    else:
        cluster_labels[c] = 'A: Awareness gap'

df['cluster_label'] = df['cluster'].map(cluster_labels)
print('\nCluster assignment:')
print(df['cluster_label'].value_counts())

In [ ]:
# --- Cluster scatter plot ---
cluster_colors = {
    'A: Awareness gap':        '#E24B4A',
    'B: Documentation barrier': '#BA7517',
    'C: Infrastructure deficit': '#534AB7',
    'D: Well-performing':       '#1D9E75',
}

fig, ax = plt.subplots(figsize=(9, 6))
for label, color in cluster_colors.items():
    sub = df[df['cluster_label'] == label]
    ax.scatter(sub['literacy_rate'], sub['uptake_gap_pct'],
               color=color, alpha=0.7, s=45, label=f'{label} (n={len(sub)})')

ax.set_xlabel('District literacy rate (%)')
ax.set_ylabel('Uptake gap (%)')
ax.set_title('District segmentation by barrier archetype')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig('fig7_cluster_scatter.png', bbox_inches='tight')
plt.show()

## 6. Anomaly Detection — Statistically Anomalous Gap Districts

Identify districts whose gap is **significantly higher than expected** given their socioeconomic indicators (residuals from OLS > 1.5σ).

In [ ]:
from sklearn.linear_model import LinearRegression

X_ols = df[features].values
y_ols = df['uptake_gap_pct'].values

reg_ols = LinearRegression().fit(X_ols, y_ols)
df['predicted_gap'] = reg_ols.predict(X_ols)
df['residual']      = df['uptake_gap_pct'] - df['predicted_gap']

residual_std = df['residual'].std()
df['anomaly'] = df['residual'] > 1.5 * residual_std

anomalous = df[df['anomaly']].sort_values('residual', ascending=False)
print(f'Anomalous districts (gap > predicted + 1.5σ): {len(anomalous)}')
print(anomalous[['district','state','uptake_gap_pct','predicted_gap','residual',
                  'literacy_rate','female_lfpr']].head(15).round(2).to_string(index=False))

In [ ]:
# --- Residual plot ---
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df['predicted_gap'], df['residual'],
           c=['#E24B4A' if a else '#B4B2A9' for a in df['anomaly']],
           alpha=0.65, s=35)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(1.5 * residual_std, color='#E24B4A', linewidth=1, linestyle='--',
           label=f'+1.5σ threshold = {1.5*residual_std:.1f}%')
ax.axhline(-1.5 * residual_std, color='#1D9E75', linewidth=1, linestyle='--',
           label=f'-1.5σ threshold')
ax.set_xlabel('Predicted uptake gap (%)')
ax.set_ylabel('Residual (actual − predicted)')
ax.set_title('OLS residuals — red dots are anomalously high-gap districts')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig8_residuals.png', bbox_inches='tight')
plt.show()

## 7. Bottom-10 Districts — Prioritised Intervention Table

In [ ]:
# Priority score: weighted composite
# Weight: gap (0.4) + days gap (0.3) + anomaly residual (0.2) + distance (0.1)
df['days_gap']       = 100 - df['avg_days_worked']
df['priority_score'] = (
    0.40 * df['uptake_gap_pct'] +
    0.30 * df['days_gap'] +
    0.20 * df['residual'].clip(lower=0) +
    0.10 * df['distance_to_gp_km'] * 3
).round(2)

bottom10 = df.nlargest(10, 'priority_score')[
    ['district','state','uptake_gap_pct','avg_days_worked','priority_score',
     'cluster_label','literacy_rate','female_lfpr','distance_to_gp_km','aadhaar_link_pct']
].reset_index(drop=True)
bottom10.index = bottom10.index + 1

print('=== Bottom-10 Priority Districts ===')
print(bottom10.to_string())

bottom10.to_csv('bottom10_priority_districts.csv')
print('\nSaved: bottom10_priority_districts.csv')

In [ ]:
# --- Priority intervention recommendations ---
intervention_map = {
    'A: Awareness gap':         'Community radio + Gram Rozgar Sahayak door-to-door drives; ASHA co-enrollment model',
    'B: Documentation barrier': 'Mobile Aadhaar-seeding camps; Common Service Centre documentation drives',
    'C: Infrastructure deficit': 'Offline NMR app + India Post Payments Bank; worksite proximity mapping',
    'D: Well-performing':        'Sustain + monitor; replicate model to neighbouring blocks',
}

bottom10['recommended_intervention'] = bottom10['cluster_label'].map(intervention_map)

print('\n=== Evidence-backed interventions for bottom-10 districts ===')
for _, row in bottom10.iterrows():
    print(f"\n{int(_)}. {row['district']} ({row['state']})")
    print(f"   Gap: {row['uptake_gap_pct']}% | Days: {row['avg_days_worked']} | Barrier: {row['cluster_label']}")
    print(f"   Action: {row['recommended_intervention']}")

In [ ]:
# --- Priority map visualization (matplotlib heatmap proxy) ---
fig, ax = plt.subplots(figsize=(11, 5))

bar_colors = {
    'A: Awareness gap':         '#E24B4A',
    'B: Documentation barrier': '#BA7517',
    'C: Infrastructure deficit': '#534AB7',
    'D: Well-performing':        '#1D9E75',
}
colors = [bar_colors.get(l, '#888') for l in bottom10['cluster_label']]

bars = ax.bar(range(len(bottom10)), bottom10['priority_score'], color=colors,
              edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(bottom10)))
ax.set_xticklabels(
    [f"{row['district']}\n({row['state'][:4]})" for _, row in bottom10.iterrows()],
    fontsize=8, rotation=30, ha='right'
)
ax.set_ylabel('Composite priority score')
ax.set_title('Bottom-10 districts by composite priority score')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in bar_colors.items()]
ax.legend(handles=legend_elements, fontsize=8, loc='upper right')

# Annotate gap %
for i, (_, row) in enumerate(bottom10.iterrows()):
    ax.text(i, row['priority_score'] + 0.4, f"{row['uptake_gap_pct']}%",
            ha='center', va='bottom', fontsize=8, color='#444')

plt.tight_layout()
plt.savefig('fig9_priority_map.png', bbox_inches='tight')
plt.show()
print('Figure 9 saved.')

## 8. GeoPandas Choropleth (India Districts)

This cell requires:
1. India district shapefile — download from: https://github.com/datameet/maps/tree/master/Districts
2. `pip install geopandas folium mapclassify`

The code below merges uptake gap data with the shapefile and produces a choropleth.

In [ ]:
# --- PRODUCTION CHOROPLETH ---
# Requires: India_Districts.shp (from datameet/maps on GitHub)

SHAPEFILE_PATH = 'India_Districts.shp'   # ← update path

try:
    import geopandas as gpd

    gdf = gpd.read_file(SHAPEFILE_PATH)
    print(f'Shapefile loaded: {len(gdf)} districts')
    print('Columns:', gdf.columns.tolist())

    # Merge on district name — adjust column name to match your shapefile
    # Common column names: 'DISTRICT', 'dtname', 'Dist_Name'
    # df column: 'district' (replace with actual district names in production)
    merge_col = 'DISTRICT'   # ← update to match your shapefile
    gdf_merged = gdf.merge(df[['district', 'uptake_gap_pct', 'cluster_label',
                                'priority_score', 'avg_days_worked']],
                           left_on=merge_col, right_on='district', how='left')

    fig, axes = plt.subplots(1, 2, figsize=(16, 9))

    # Choropleth 1: uptake gap
    gdf_merged.plot(
        column='uptake_gap_pct', cmap='RdYlGn_r', linewidth=0.3,
        edgecolor='white', legend=True, missing_kwds={'color': '#eee'},
        ax=axes[0], legend_kwds={'label': 'Uptake gap (%)', 'shrink': 0.6}
    )
    axes[0].set_title('MGNREGS uptake gap by district', fontsize=11)
    axes[0].axis('off')

    # Choropleth 2: average days worked
    gdf_merged.plot(
        column='avg_days_worked', cmap='YlGn', linewidth=0.3,
        edgecolor='white', legend=True, missing_kwds={'color': '#eee'},
        ax=axes[1], legend_kwds={'label': 'Avg days worked', 'shrink': 0.6}
    )
    axes[1].set_title('MGNREGS average days worked per HH', fontsize=11)
    axes[1].axis('off')

    plt.suptitle('MGNREGS District-level Analysis (FY 2023-24)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('fig10_choropleth.png', bbox_inches='tight', dpi=150)
    plt.show()

    # Interactive Folium map
    import folium
    import json

    m = folium.Map(location=[22.5, 82.5], zoom_start=5, tiles='CartoDB positron')
    folium.Choropleth(
        geo_data=gdf_merged.to_json(),
        data=gdf_merged,
        columns=[merge_col, 'uptake_gap_pct'],
        key_on=f'feature.properties.{merge_col}',
        fill_color='RdYlGn_r',
        fill_opacity=0.75,
        line_opacity=0.2,
        legend_name='MGNREGS Uptake Gap (%)',
        nan_fill_color='#cccccc'
    ).add_to(m)

    # Tooltip
    folium.GeoJson(
        gdf_merged,
        tooltip=folium.GeoJsonTooltip(
            fields=[merge_col, 'uptake_gap_pct', 'avg_days_worked', 'cluster_label'],
            aliases=['District', 'Uptake gap (%)', 'Avg days worked', 'Barrier type'],
            localize=True
        ),
        style_function=lambda x: {'fillOpacity': 0, 'weight': 0}
    ).add_to(m)

    m.save('mgnregs_interactive_map.html')
    print('Interactive map saved: mgnregs_interactive_map.html')

except FileNotFoundError:
    print('Shapefile not found at:', SHAPEFILE_PATH)
    print('Download from: https://github.com/datameet/maps/tree/master/Districts')
    print('Then update SHAPEFILE_PATH and re-run this cell.')
except ImportError as e:
    print(f'Missing library: {e}')
    print('Install: pip install geopandas folium mapclassify')

## 9. Summary: Key Findings & Recommendations

In [ ]:
print('=' * 65)
print('MGNREGS PARTICIPATION ANALYSIS — KEY FINDINGS SUMMARY')
print('=' * 65)

print(f"""
1. NATIONAL GAP
   Avg uptake gap: {df['uptake_gap_pct'].mean():.1f}%  |  Avg days worked: {df['avg_days_worked'].mean():.1f}/100
   Worst state:    {df.groupby('state')['uptake_gap_pct'].mean().idxmax()} ({df.groupby('state')['uptake_gap_pct'].mean().max():.1f}% gap)
   Best state:     {df.groupby('state')['uptake_gap_pct'].mean().idxmin()} ({df.groupby('state')['uptake_gap_pct'].mean().min():.1f}% gap)

2. KEY PREDICTORS (OLS regression, R²={r2_score(y, reg_std.predict(X_scaled)):.2f})
   Strongest positive predictor of gap: Distance to GP (β={coef_std['distance_to_gp_km']:.2f})
   Strongest negative predictor:        Literacy rate  (β={coef_std['literacy_rate']:.2f})
   Gender signal: Female LFPR           (β={coef_std['female_lfpr']:.2f})
   Aadhaar linkage:                     (β={coef_std['aadhaar_link_pct']:.2f})

3. ANOMALOUS DISTRICTS
   {len(anomalous)} districts have gaps >1.5σ above what their socioeconomic profile predicts.
   These are priority candidates for governance/corruption audit.

4. CLUSTER BREAKDOWN
""")
print(df['cluster_label'].value_counts().to_string())

print(f"""
5. EVIDENCE-BACKED RECOMMENDATIONS

   TYPE A — Awareness gap districts:
     → Community radio IEC campaigns (J-PAL: +29% demand in Andhra AP pilot)
     → ASHA/ANM co-enrollment; Gram Rozgar Sahayak reactivation
     → Gram Sabha demand-activation drives with community mobilisers

   TYPE B — Documentation barrier districts:
     → 30-day Aadhaar-seeding mobile camps at panchayat level
     → Forest Rights Act + job card issuance integration (tribal areas)
     → Post Office / Jan Aushadhi Jan Seva Kendra as documentation points

   TYPE C — Infrastructure deficit districts:
     → Offline-capable NMR app deployment (MoRD prototype exists)
     → India Post Payments Bank for last-mile wage delivery
     → Worksite proximity mapping; satellite GP option for remote hamlets

   CROSS-CUTTING:
     → Women-only facilitation groups for work demand submission
     → Worksite creches + transport allowance for female workers
     → Mandatory biannual social audits with external teams

6. DATA GAPS TO ADDRESS
   → Real district-level Aadhaar linkage data (MoRD MIS, not estimated)
   → NFHS-5 female LFPR matched to MGNREGS district codes
   → Reason codes for non-enrollment (survey or grievance portal analysis)
   → Wage payment delay data (days from work completion to credit)
""")
print('=' * 65)

---
## Reproducibility & Data Sources

| Dataset | Source | URL |
|---|---|---|
| MGNREGS MIS | Ministry of Rural Development | https://mnregaweb4.nic.in |
| Census 2011 PCA | Census of India | https://censusindia.gov.in |
| NFHS-5 District | IIPS / MoHFW | https://rchiips.org/nfhs |
| data.gov.in MGNREGS | MoRD / NIC | https://data.gov.in |
| India Districts Shapefile | DataMeet | https://github.com/datameet/maps |

**References:**  
- J-PAL South Asia (2019). *Improving MGNREGA Implementation.*  
- IDFC Institute (2021). *MGNREGS Wage Payment Delays and Worker Outcomes.*  
- CAG Report No. 7 (2023). *Performance Audit of MGNREGS.*  
- CPR India (2022). *Social Audit and Leakage Reduction in MGNREGS.*

**License:** MIT — reproduce freely with attribution.